# Business Understanding

## Background

* The AIC Kijabe hospital's outpatient department (OPD) currently manages patient flow across approximately forty departments (General OPD, Casualty, Renal, Oncology, and others) without a structured way to anticipate periods of high patient volume. This can lead to under-staffing during surges and inefficient resource allocation during quieter periods.

## Business Objectives

* Enable the hospital to anticipate periods of higher patient volume in advance, so that staffing and resources can be planned proactively rather than reactively.
* Understand whether seasonal patterns (i.e rainy vs dry seasons in Kenya) affect patient volume, to support longer-term capacity planning.
* Understand patient return behavior (how long patients typically go before returning to the hospital) to support follow-up care planning and identify departments with outlying short or long return intervals.

## Data Analysis Goals

* Build a time series forecasting model to predict daily department-level patient arrival volume.
* Quantify the relationship between seasonal patterns and patient volume, including any lagged effects.
* Apply survival analysis to model time-to-return-visit, since discharge/exit data is not available in this dataset, return-visit interval is used as a signal for patient care continuity.

## Success Criteria

* Surge model - Forecast accuracy to be more useful than a simple "same as last week" baseline (e.g., a measurable improvement in MAE/RMSE).
* Seasonal correlation - A clear, evidence-based statement on whether seasons have a meaningful effect on patient volume.
* Survival analysis - Identifiable differences in return-visit patterns across at least one patient segment (e.g., department, age group).

# Data Preparation

In [30]:
# %pip install openpyxl

In [31]:
# import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [32]:
# load the data
data = pd.read_excel("Opd_data.xlsx")
data.head()

,PatientNumber,RegistrationDate,Gender,Age,QueuedTo,ConsultDescription
0,010575758,2024-06-01 00:00:00.000,Male,38 Yr(s),GENERAL OPD,GEneral Outpatient Care( NEW )
1,010575713,2024-06-01 00:00:00.000,Female,29 Yr(s),NaN,NaN
2,010569291,2024-06-01 00:04:38.083,Female,44 Yr(s),GENERAL OPD,General Outpatient Care
3,010575732,2024-06-01 00:04:42.460,Female,78 Yr(s),GENERAL OPD,General Outpatient Care
4,010575731,2024-06-01 00:10:16.037,Female,2 Yr(s),ADMISSION,Admission


In [33]:
# Basic exploratory
data.shape
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 306306 entries, 0 to 306305
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   PatientNumber       306306 non-null  str           
 1   RegistrationDate    306306 non-null  datetime64[us]
 2   Gender              306291 non-null  str           
 3   Age                 305735 non-null  str           
 4   QueuedTo            299528 non-null  str           
 5   ConsultDescription  299920 non-null  str           
dtypes: datetime64[us](1), str(5)
memory usage: 14.0 MB
None


## Data Cleaning

In [34]:
# Check for missing values
data.isnull().sum()

PatientNumber            0
RegistrationDate         0
Gender                  15
Age                    571
QueuedTo              6778
ConsultDescription    6386
dtype: int64

In [35]:
# check for duplicates
data.duplicated().sum()

355

In [36]:
# Drop the duplicates
data= data.drop_duplicates()

In [37]:
# Recheck for duplicates
data.duplicated().sum()

0

In [38]:
# Drop gender null values
data = data[data['Gender'].notna()]

In [39]:
# Standardize casing
data['Gender'] = data['Gender'].str.strip().str.upper()

## Age In Years

In [40]:
# extract the number and the unit separately
data['AgeNum'] = data['Age'].str.extract(r'(\d+)').astype(float)
# the numeric part, 38
data['AgeUnit'] = data['Age'].str.extract(r'\d+\s*(\D+)')[0].str.strip()  
# the unit part, "Yr(s)"

# convert based on unit: years stay the same, months divide by 12, weeks divide by 52, days divide by 365
conditions = [
    data['AgeUnit'] == 'Yr(s)',
    data['AgeUnit'] == 'Mth(s)',
    data['AgeUnit'] == 'Week(s)',
    data['AgeUnit'] == 'Days(s)'
]
choices = [
    data['AgeNum'],
    data['AgeNum'] / 12,
    data['AgeNum'] / 52,
    data['AgeNum'] / 365
]
data['AgeYears'] = np.select(conditions, choices, default=np.nan)

# round to 2 decimal places
data['AgeYears'] = data['AgeYears'].round(2)  

# only interested in age years
data = data.drop(columns=['AgeNum', 'AgeUnit'])


In [41]:
# Turning all impossible ages (>100)into NAN
data.loc[data['AgeYears'] > 100, 'AgeYears'] = np.nan

In [42]:
# calculate the mean age from all remaining valid values (this naturally excludes 
# both the >100 values we just nulled and any age that was already missing)
mean_age = round(data['AgeYears'].mean(), 2)
print(round(mean_age, 2))

40.86


In [43]:
# impute missing ages with mean
data['AgeYears'] = data['AgeYears'].fillna(mean_age)

print(mean_age)
# print(data['Age_num'].isna().sum())
print(data['AgeYears'].isna().sum())

40.86
0


In [44]:
# Bin into ages groups
data['AgeGroup']  = pd.cut(
    data['AgeYears'],                                    
    bins=[0,5,12,18,35,60,100],
    labels=['Infant(0-5)','Child(6-12)','Teen(13-18)','Young Adult(19-35)','Adult(36-60)','Senior(61-100)'],
    include_lowest=True
)

In [45]:
# Drop missing QueuedTo
data = data.dropna(subset=['QueuedTo'])

In [46]:
# Clean the QueuedTo column
data['QueuedTo']= data['QueuedTo'].str.strip().str.upper()

In [47]:
# Drop the age column
data = data.drop(columns=['Age'])

In [48]:
# Rechecking the null values
data.isna().sum()

PatientNumber         0
RegistrationDate      0
Gender                0
QueuedTo              0
ConsultDescription    0
AgeYears              0
AgeGroup              0
dtype: int64

In [49]:
data[data['AgeGroup'].isna()]['AgeYears'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: AgeYears, dtype: float64

In [50]:
data['AgeGroup'].unique()

['Adult(36-60)', 'Senior(61-100)', 'Infant(0-5)', 'Young Adult(19-35)', 'Child(6-12)', 'Teen(13-18)']
Categories (6, str): ['Infant(0-5)' < 'Child(6-12)' < 'Teen(13-18)' < 'Young Adult(19-35)' < 'Adult(36-60)' < 'Senior(61-100)']

In [51]:
# Recheck everything
print("Shape:", data.shape)
print()
print("Missing values:")
print(data.isnull().sum())
print()
print("Duplicate rows:", data.duplicated().sum())
print()
print("Dtypes:")
print(data.dtypes)
print()
print("Age range check:", data['AgeYears'].min(), "-", data['AgeYears'].max())
## print("Age range check:", data['Age_num'].min(), "-", data['Age_num'].max())
print()
print("Columns:", list(data.columns))

Shape: (299162, 7)

Missing values:
PatientNumber         0
RegistrationDate      0
Gender                0
QueuedTo              0
ConsultDescription    0
AgeYears              0
AgeGroup              0
dtype: int64

Duplicate rows: 0

Dtypes:
PatientNumber                    str
RegistrationDate      datetime64[us]
Gender                           str
QueuedTo                         str
ConsultDescription               str
AgeYears                     float64
AgeGroup                    category
dtype: object

Age range check: 0.0 - 100.0

Columns: ['PatientNumber', 'RegistrationDate', 'Gender', 'QueuedTo', 'ConsultDescription', 'AgeYears', 'AgeGroup']


In [52]:
# Exploring the data
data['QueuedTo'].unique()

<StringArray>
[                        'GENERAL OPD',                           'ADMISSION',
                            'CASUALTY',                        'GEN SURG OPD',
                               'RENAL',                                 'MCH',
                       'PHYSIOTHERAPY',                             'DAYCASE',
                              'DENTAL',                   'SPECIALITY CLINIC',
                          'PSYCHOLOGY',                                'OHNS',
                            'ONCOLOGY',                      'PRIVATE CLINIC',
                          'EYE CLINIC',                          'PAEDS BKKH',
 'CHRONIC CARE CLINIC (DM/HTN/TB/CCC)',                           'ORTHO OPD',
                          'PALLIATIVE',                           'AUDIOLOGY',
                           'NUTRITION',                'OCCUPATIONAL THERAPY',
          'NEURO SURGERY CONSULTATION',                     'FAMILY MEDICINE',
                     'DIABETIC CLINIC'